## Theory

In [1]:
class BankAccount {
    private final String accountNumber;  // final — cannot change
    private double balance;
    private boolean frozen;
    
    private static final double MIN_BALANCE = 1000.0;
    
    public BankAccount(String accountNumber, double initialDeposit) {
        if (initialDeposit < MIN_BALANCE)
            throw new IllegalArgumentException("Min deposit: INR " + MIN_BALANCE);
        this.accountNumber = accountNumber;
        this.balance = initialDeposit;
        this.frozen = false;
    }
    
    public boolean deposit(double amount) {
        if (frozen || amount <= 0) return false;
        balance += amount;
        System.out.println("Deposited: INR " + amount + " | Balance: INR " + balance);
        return true;
    }
    
    public boolean withdraw(double amount) {
        if (frozen || amount <= 0 || balance - amount < MIN_BALANCE) return false;
        balance -= amount;
        System.out.println("Withdrawn: INR " + amount + " | Balance: INR " + balance);
        return true;
    }
    
    // Getter only — no setter for balance (controlled via deposit/withdraw)
    public double getBalance() { return balance; }
    public String getAccountNumber() { return accountNumber; }
}

BankAccount acc = new BankAccount("ACC-001", 5000);
acc.deposit(2000);
acc.withdraw(1000);
acc.withdraw(10000); // Fails — min balance protection
System.out.println("Final balance: INR " + acc.getBalance());

Deposited: INR 2000.0 | Balance: INR 7000.0
Withdrawn: INR 1000.0 | Balance: INR 6000.0
Final balance: INR 6000.0


## Real-World Encapsulation

In [2]:
// Immutable class example
final class Money {
    private final double amount;
    private final String currency;
    
    public Money(double amount, String currency) {
        if (amount < 0) throw new IllegalArgumentException("Amount cannot be negative");
        this.amount = amount;
        this.currency = currency;
    }
    
    public Money add(Money other) {
        if (!this.currency.equals(other.currency))
            throw new IllegalStateException("Currency mismatch");
        return new Money(this.amount + other.amount, this.currency); // returns new object!
    }
    
    public double getAmount() { return amount; }
    public String getCurrency() { return currency; }
    public String toString() { return currency + " " + String.format("%.2f", amount); }
}

Money price = new Money(999.99, "INR");
Money tax = new Money(180.0, "INR");
Money total = price.add(tax);
System.out.println("Price: " + price);
System.out.println("Tax  : " + tax);
System.out.println("Total: " + total);

Price: INR 999.99
Tax  : INR 180.00
Total: INR 1179.99


## Mini Challenge
Design a `MedicalRecord` class that encapsulates patient data. Ensure date of birth cannot be changed, diagnosis list returns a copy, and blood group is validated.

In [3]:
import java.time.LocalDate;
import java.util.ArrayList;
import java.util.Collections;
import java.util.List;
import java.util.Objects;

public final class MedicalRecord {
    
    // 1. Encapsulated fields
    private final String patientId;
    private final String patientName;
    private final LocalDate dateOfBirth; // Immutable after initialization
    private BloodGroup bloodGroup;       // Validated via Enum
    private final List<String> diagnoses; // Internal mutable list

    // 2. Constructor with validation
    public MedicalRecord(String patientId, String patientName, LocalDate dateOfBirth, BloodGroup bloodGroup) {
        this.patientId = Objects.requireNonNull(patientId, "Patient ID cannot be null");
        this.patientName = Objects.requireNonNull(patientName, "Patient name cannot be null");
        
        // Ensure DOB is in the past
        if (dateOfBirth == null || dateOfBirth.isAfter(LocalDate.now())) {
            throw new IllegalArgumentException("Invalid date of birth");
        }
        this.dateOfBirth = dateOfBirth;
        
        this.bloodGroup = Objects.requireNonNull(bloodGroup, "Blood group cannot be null");
        this.diagnoses = new ArrayList<>();
    }

    // 3. Getters & Setters
    
    public String getPatientId() { return patientId; }
    public String getPatientName() { return patientName; }
    
    // Date of Birth only has a getter, satisfying "cannot be changed"
    public LocalDate getDateOfBirth() { return dateOfBirth; }

    public BloodGroup getBloodGroup() { return bloodGroup; }
    
    public void setBloodGroup(BloodGroup bloodGroup) {
        this.bloodGroup = Objects.requireNonNull(bloodGroup, "Blood group cannot be null");
    }

    /**
     * Returns an unmodifiable view or a deep copy of the diagnosis list.
     * This prevents external code from doing: record.getDiagnoses().clear()
     */
    public List<String> getDiagnoses() {
        // Alternatively: return new ArrayList<>(this.diagnoses);
        return Collections.unmodifiableList(diagnoses);
    }

    // 4. Business Logic Methods
    public void addDiagnosis(String diagnosis) {
        if (diagnosis == null || diagnosis.strip().isEmpty()) {
            throw new IllegalArgumentException("Diagnosis cannot be empty");
        }
        this.diagnoses.add(diagnosis.strip());
    }

    // 5. Blood Group Validation Enum
    public enum BloodGroup {
        A_POSITIVE("A+"), A_NEGATIVE("A-"),
        B_POSITIVE("B+"), B_NEGATIVE("B-"),
        AB_POSITIVE("AB+"), AB_NEGATIVE("AB-"),
        O_POSITIVE("O+"), O_NEGATIVE("O-");

        private final String label;
        BloodGroup(String label) { this.label = label; }
        public String getLabel() { return label; }
    }
}

In [6]:
import java.time.LocalDate;

MedicalRecord record = new MedicalRecord(
    "PAT-2026-88", 
    "Jane Doe", 
    LocalDate.of(1990, 8, 14), 
    MedicalRecord.BloodGroup.AB_NEGATIVE
);

record.addDiagnosis("Hypertension");
record.addDiagnosis("Seasonal Allergies");

// Jupyter will automatically print the evaluation of the last line if you don't use a semicolon
record.getDiagnoses()

[Hypertension, Seasonal Allergies]